In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import yaml

from credit_risk.pipelines.data_preprocess import (
    build_master_dataset,
    build_origination,
    build_performance,
)
from credit_risk.utils.config import create_path
import os
# Fix: Use .parent directly on the Path object
PROJECT_ROOT = Path.cwd().resolve().parent
PARAMETERS_PATH = PROJECT_ROOT / "config" / "parameters" / "base.yml"
os.chdir(PROJECT_ROOT)
CATALOG_PATH = PROJECT_ROOT / "config" / "catalog" / "base.yml"

with PARAMETERS_PATH.open("r", encoding="utf-8") as file:
    parameters = yaml.safe_load(file)

with CATALOG_PATH.open("r", encoding="utf-8") as file:
    catalog = yaml.safe_load(file)

config = {
    **parameters,
    **catalog,
}

data_config = config["parameters"]["data"]
behavioral_config = config["parameters"]["behavioral"]

provider = data_config["data_provider"]

print("Project root:", PROJECT_ROOT)
print("Provider:", provider)
print(
    "Observation ages:",
    behavioral_config["observation_ages"],
)

Project root: C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk
Provider: freddie_mac
Observation ages: [6, 12]


In [2]:
VINTAGE = 2015

print("Debug vintage:", VINTAGE)

Debug vintage: 2015


In [3]:
origination_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "origination_path",
    provider,
    VINTAGE,
)

performance_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "performance_path",
    provider,
    VINTAGE,
)

print("Origination path:")
print(origination_path)

print("\nPerformance path:")
print(performance_path)

Origination path:
data\02_intermediate\freddie_mac\2015\origination.parquet

Performance path:
data\02_intermediate\freddie_mac\2015\performance


In [4]:
origination_df = pd.read_parquet(
    origination_path,
)

performance_df = pd.read_parquet(
    performance_path,
)

print(
    "Origination:",
    f"{len(origination_df):,}",
    "rows x",
    f"{origination_df.shape[1]:,}",
    "columns",
)

print(
    "Performance:",
    f"{len(performance_df):,}",
    "rows x",
    f"{performance_df.shape[1]:,}",
    "columns",
)

Origination: 50,000 rows x 31 columns
Performance: 3,467,166 rows x 35 columns


In [26]:
origination_features = build_origination(
    origination_df,
    config,
)

# Keep first_payment_date for the V2 behavioural time axis.
origination_dates = origination_df[
    [
        "loan_id",
        "first_payment_date",
    ]
].drop_duplicates(
    subset=["loan_id"],
)

origination_features = origination_features.merge(
    origination_dates,
    on="loan_id",
    how="left",
    validate="one_to_one",
)

performance_features = build_performance(
    performance_df,
)

master = build_master_dataset(
    origination_features,
    performance_features,
)

print(
    "Master:",
    f"{len(master):,}",
    "rows x",
    f"{master.shape[1]:,}",
    "columns",
)

print(
    "first_payment_date present:",
    "first_payment_date" in master.columns,
)

display(
    master[
        [
            "loan_id",
            "first_payment_date",
            "period",
            "loan_age",
        ]
    ].head(20)
)

Master: 3,467,166 rows x 31 columns
first_payment_date present: True


,loan_id,first_payment_date,period,loan_age
0,F15Q10000025,2015-04,2015-03,0
1,F15Q10000025,2015-04,2015-04,1
2,F15Q10000025,2015-04,2015-05,2
3,F15Q10000025,2015-04,2015-06,3
4,F15Q10000025,2015-04,2015-07,4
5,F15Q10000025,2015-04,2015-08,5
6,F15Q10000025,2015-04,2015-09,6
7,F15Q10000025,2015-04,2015-10,7
8,F15Q10000025,2015-04,2015-11,8
9,F15Q10000025,2015-04,2015-12,9


In [27]:
print(
    "first_payment_date dtype:",
    master["first_payment_date"].dtype,
)

print(
    "period dtype:",
    master["period"].dtype,
)

display(
    master[
        [
            "loan_id",
            "first_payment_date",
            "period",
        ]
    ].head(10)
)

first_payment_date dtype: period[M]
period dtype: period[M]


,loan_id,first_payment_date,period
0,F15Q10000025,2015-04,2015-03
1,F15Q10000025,2015-04,2015-04
2,F15Q10000025,2015-04,2015-05
3,F15Q10000025,2015-04,2015-06
4,F15Q10000025,2015-04,2015-07
5,F15Q10000025,2015-04,2015-08
6,F15Q10000025,2015-04,2015-09
7,F15Q10000025,2015-04,2015-10
8,F15Q10000025,2015-04,2015-11
9,F15Q10000025,2015-04,2015-12


In [30]:
# Convert both fields to string YYYY-MM values first
master["period_month"] = master["period"].astype(str)

master["first_payment_month"] = master["first_payment_date"].astype(str)

# Extract year and month
period_year = master["period_month"].str[:4].astype(int)
period_month = master["period_month"].str[5:7].astype(int)

first_payment_year = master["first_payment_month"].str[:4].astype(int)

first_payment_month = master["first_payment_month"].str[5:7].astype(int)

# Calculate elapsed months
master["calculated_loan_age"] = (period_year - first_payment_year) * 12 + (
    period_month - first_payment_month
) +1

display(
    master[
        [
            "loan_id",
            "first_payment_date",
            "period",
            "loan_age",
            "calculated_loan_age",
        ]
    ].head(20)
)

,loan_id,first_payment_date,period,loan_age,calculated_loan_age
0,F15Q10000025,2015-04,2015-03,0,0
1,F15Q10000025,2015-04,2015-04,1,1
2,F15Q10000025,2015-04,2015-05,2,2
3,F15Q10000025,2015-04,2015-06,3,3
4,F15Q10000025,2015-04,2015-07,4,4
5,F15Q10000025,2015-04,2015-08,5,5
6,F15Q10000025,2015-04,2015-09,6,6
7,F15Q10000025,2015-04,2015-10,7,7
8,F15Q10000025,2015-04,2015-11,8,8
9,F15Q10000025,2015-04,2015-12,9,9


In [31]:
calculated_age_duplicates = master.loc[
    master.duplicated(
        subset=[
            "loan_id",
            "calculated_loan_age",
        ],
        keep=False,
    )
].sort_values(
    [
        "loan_id",
        "calculated_loan_age",
        "period",
    ]
)

print("Duplicate loans:", calculated_age_duplicates["loan_id"].nunique())

display(calculated_age_duplicates.head(50))

Duplicate loans: 0


,loan_id,period,current_actual_upb,current_interest_rate,loan_age,remaining_months_to_legal_maturity,estimated_ltv,current_loan_delinquency_status,ddlpi,zero_balance_code,...,super_conforming_flag,harp_indicator,property_state,msa,original_dti_missing,first_payment_date,first_payment_date_dt,period_month,first_payment_month,calculated_loan_age


In [38]:
model_input_path = create_path("data", catalog["catalog"], "model_input_path","behavioral","freddie_mac","2015")

In [39]:
behavioral_features = pd.read_parquet(model_input_path)

print(behavioral_features.shape)

print(behavioral_features["observation_age"].value_counts().sort_index())

print(
    "Unique loans:",
    behavioral_features["loan_id"].nunique(),
)

print(
    "Duplicate loan_id × observation_age:",
    behavioral_features.duplicated(
        subset=[
            "loan_id",
            "observation_age",
        ]
    ).sum(),
)

(92950, 33)
observation_age
6     47604
12    45346
Name: count, dtype: int64
Unique loans: 47643
Duplicate loan_id × observation_age: 0


In [40]:
# ============================================================
# V2 FINAL MODELING DATASET QC
# ============================================================

print("=" * 70)
print("1. DATASET SHAPE")
print("=" * 70)

print(f"Rows:    {len(modeling):,}")
print(f"Columns: {modeling.shape[1]:,}")


print("\n" + "=" * 70)
print("2. OBSERVATION AGE DISTRIBUTION")
print("=" * 70)

display(
    modeling["observation_age"]
    .value_counts()
    .sort_index()
    .rename_axis("observation_age")
    .reset_index(name="rows")
)


print("\n" + "=" * 70)
print("3. OBSERVATIONS / LOANS / EVENTS BY AGE")
print("=" * 70)

age_summary = (
    modeling.groupby("observation_age")
    .agg(
        observations=("loan_id", "size"),
        unique_loans=("loan_id", "nunique"),
        events=("future_90dpd_12m", "sum"),
        event_rate=("future_90dpd_12m", "mean"),
    )
    .reset_index()
)

display(age_summary)


print("\n" + "=" * 70)
print("4. DUPLICATE loan_id × observation_age")
print("=" * 70)

duplicate_count = int(
    modeling.duplicated(
        subset=[
            "loan_id",
            "observation_age",
        ],
    ).sum()
)

print(f"Duplicate rows: {duplicate_count:,}")


print("\n" + "=" * 70)
print("5. CALCULATED AGE VS OBSERVATION AGE")
print("=" * 70)

age_mismatch_count = int(
    (modeling["calculated_loan_age"] != modeling["observation_age"]).sum()
)

print(f"Age mismatches: {age_mismatch_count:,}")


print("\n" + "=" * 70)
print("6. TARGET VALIDITY")
print("=" * 70)

print(
    "Missing target:",
    int(modeling["future_90dpd_12m"].isna().sum()),
)

print(
    "Unique target values:",
    sorted(modeling["future_90dpd_12m"].dropna().unique().tolist()),
)


print("\n" + "=" * 70)
print("7. LOAN OVERLAP ACROSS OBSERVATION AGES")
print("=" * 70)

loans_by_age = modeling.groupby("loan_id")["observation_age"].nunique()

print(
    "Loans appearing at both observation ages:",
    int(loans_by_age.eq(len(modeling["observation_age"].unique())).sum()),
)

print(
    "Loans appearing at only one observation age:",
    int(loans_by_age.eq(1).sum()),
)


print("\n" + "=" * 70)
print("8. REQUIRED COLUMNS")
print("=" * 70)

required_columns = {
    "loan_id",
    "period",
    "first_payment_date",
    "calculated_loan_age",
    "observation_age",
    "future_90dpd_12m",
}

missing_columns = sorted(required_columns - set(modeling.columns))

print(
    "Missing required columns:",
    missing_columns if missing_columns else "None",
)


print("\n" + "=" * 70)
print("9. PASS / FAIL")
print("=" * 70)

checks = {
    "No duplicate loan_id × observation_age": duplicate_count == 0,
    "No calculated age mismatches": age_mismatch_count == 0,
    "No missing target": modeling["future_90dpd_12m"].isna().sum() == 0,
    "Target is binary": set(modeling["future_90dpd_12m"].dropna().unique()).issubset(
        {0, 1}
    ),
    "Required columns present": len(missing_columns) == 0,
}

for check, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {check}")

1. DATASET SHAPE


NameError: name 'modeling' is not defined